observations:
- depth sensor
- compass heading
- velocity

state: 

$$
x = 
\begin{bmatrix}
p_1 \\ p_2 \\ v_1 \\ v_2
\end{bmatrix}
$$

dynamics:

$$
F = 
\begin{bmatrix}
1 & 0 & \Delta t  & 0 \\
0 & 1 & 0 & \Delta t \\
0 & 0 & 1 & 0 \\
0 & 0 & 0 & 1
\end{bmatrix}
$$


Observation function
$$
h(x) = 
\begin{bmatrix}
bath(p_1, p_2) \\ tan(\frac{p_2}{p_1}) \\ v_1 \\ v_2
\end{bmatrix}
$$

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import ipywidgets as widgets
import seaborn as sns

from functools import partial
from functools import Placeholder
from scipy.interpolate import RegularGridInterpolator

sns.set_theme(context='notebook', style='darkgrid')

In [2]:
# multi-gaussian surface
def gaussian_surface(x_in, centers, A, covs):
    """
    x_in:    (N, 1, 2, 1)
    centers: (1, M, 2, 1)
    A:       (1, M)
    covs:    (1, M, 2, 2)
    """
    d = x_in - centers
    cov_inv = np.linalg.inv(covs)

    q = d.transpose(0, 1, 3, 2) @ cov_inv @ d
    q = q.squeeze((-1, -2))

    heights = A * np.exp(-0.5 * q)

    return heights.sum(axis=1)



from scipy.stats import multivariate_normal

def normal_pdf(x, mean=0.0, std=1.0):
    variance = std ** 2
    denominator = np.sqrt(2 * np.pi * variance)
    numerator = np.exp(-((x - mean) ** 2) / (2 * variance))
    return numerator / denominator
    

## Create Generate Simulated Bathymetric Map

In [54]:
nx, ny = (200, 200)

x = np.linspace(-1, 1, nx)
y = np.linspace(-1, 1, ny)

mesh = np.meshgrid(x, y)
xv, yv = mesh

# (N, 1, 2, 1)
coords = np.stack([xv.ravel(), yv.ravel()], axis=1)
coords = coords[:, None, :, None]

# Centers: (1, M, 2, 1)
centers = np.array([
    [[-0.3], [ 0.0]],
    [[ 0.5], [ 0.3]],
    [[ 0.0], [-0.6]],
])[None]

# Heights: (1, M)
# Positive = peak, negative = valley
A = np.array([[10.0, 7.0, -4.0]])

# One covariance matrix per peak/valley: (1, M, 2, 2)
covs = np.array([
    [[0.08, 0.00],
     [0.00, 0.04]],

    [[0.2, 0.00],
     [0.00, 0.1]],

    [[0.15, 0.00],
     [0.00, 0.05]],
])[None]

z = gaussian_surface(coords, centers, A, covs).reshape(ny, nx)

## Visualize Bathymetric Map and Likelihood

In [23]:
get_depth = RegularGridInterpolator((x, y), z)
likelihood = lambda obs: normal_pdf(obs, z)

# Set heatmap axis labels
xtick_ixs = np.concat([np.arange(0, nx, 20), [nx - 1]])
xtick_labels = [f"{v:.2f}" for v in x[xtick_ixs]]

ytick_ixs = np.concat([np.arange(0, ny, 20), [ny - 1]])
ytick_labels = [f"{v:.2f}" for v in y[::-1][ytick_ixs]]

# Define a truncated color palette (10% less blacks)
base = plt.get_cmap("mako")
p = colors.LinearSegmentedColormap.from_list(
    "mako_truncated",
    base(np.linspace(0.1, 0.95, 256))  # truncate lower and upper color bounds
)

min_depth = z.min()
max_depth = z.max()
@widgets.interact(
    observation=widgets.FloatSlider(value=min_depth, min=min_depth, max=max_depth, description="Observation Depth", layout=widgets.Layout(width="60%"), style={"description_width": "15%"})
)
def likelihood_widget(observation):
    
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    
    sns.heatmap(z, cmap=p, ax=ax[0])
    sns.heatmap(likelihood(observation), ax=ax[1], xticklabels=False, yticklabels=False)

    ax[0].set_xticks(xtick_ixs, labels=xtick_labels, rotation=45)
    ax[0].set_yticks(ytick_ixs, labels=ytick_labels, rotation=0)
    ax[0].invert_yaxis()
    ax[1].invert_yaxis()

    ax[0].contour(
        z,
        levels=25,
        colors='k',
        linewidths=1,
        alpha=0.4
    )

interactive(children=(FloatSlider(value=-3.8833661844401037, description='Observation Depth', layout=Layout(wi…

## Notes

So I'm working out how to calculate transition probabilities if I'm representing spacial probabilities on a grid. 

I know that X_new = F * X_old + error, with error ~ Normal(0, Q), 

That means X_new ~ N(F * X_old, Q)

To represent this probability, I would simply multiply F by every potential state. I have every potential coordinate in my numpy meshgrid, but I would need to append the velocity to each coordinate.  So, what I need to do is expand my np.meshgrid to have [vx, vy] as well. 

state is 4 dimensional. then belief represents a probability for every possible value of every state: (x, y, v_x, v_y). The possible position states are the pixel locations on the map. 

So belief will be 4 dimensional, with shape (H, W, Nv_x, Nv_y) since the possible x states is the number of pixels of width, and y states are the number of pixels of height. 

I do not know how to represent the velocity states, because velocity is assumed constant. I guess I should just pick a range, maybe zero to one?

In [53]:
# belief distribution over all possible positions and velocities. 
vx0 = 0.1
vy0 = 0.1

# Resolution of bathymetry map
W = 200
H = 200

# Assume uniform distribution for positions on map
prior_x = np.ones(W) / W
prior_y = np.ones(H) / H

# Assume a gaussian near our target velocities 
vx = np.linspace(0, 1, 10)
vy = np.linspace(0, 1, 10)
prior_vx = normal_pdf(vx, vx0, std=1e-2)
prior_vy = normal_pdf(vy, vy0, std=1e-2)

# belief is of shape (H, W, N_vx, N_vy)
prior = np.prod(np.stack(np.meshgrid(prior_x, prior_y, prior_vx, prior_vy)), axis=0)
print(prior.shape)

(200, 200, 10, 10)


In [52]:
# p(x_new | x_old) transition probability
# X_new = F * X_old + e, e ~ N(0, Q) 
# --> X_new ~ N(Fx_old, Q)

# fixed timestep and dynamics for constant velocity
dt = 0.5
F = np.array([
    [1, 0, dt, 0],
    [0, 1, 0, dt],
    [0, 0, 1, 0],
    [0, 0, 0, 1],
], dtype=float)

# But need Fx_old for all possible values of x. 
x = np.linspace(-1, 1, W)
y = np.linspace(-1, 1, H)

np.stack(np.meshgrid(x, y, vx, vy), axis=0).shape, F.shape

# There are 200*200*10*10 potential state combinations. We want the probability of going from any one of them to any other one of them. That matrix should be huge... 
#(4, 4) x (4, 200, 200, 10, 10) ->

transition_distribution = F @ np.stack(np.meshgrid(x, y, vx, vy), axis=0)

(4, 1, 200, 200, 10, 10)

In [33]:
class linear_model:
    def __init__(self, F: np.ndarray , Q: np.ndarray = [[0, 0], [0, 0]]):
        self.F = F 
        self.Q = Q
        
    def __call__(self, x_old: np.ndarray):
        x_new = self.F @ x_old + np.random.multivariate_normal(mean=[0, 0], cov=self.Q, size=1).T
        return x_new


# transition matrix with constant velocity


# process error covariance
cov = np.eye(like=F) * 1e-4

# derive transition probability distribution
# X_new = F * X_old + error, with constant v ----> N(F * x_old, sigma_error)

p = multivariate_normal.pdf(
    mesh,
    mean=mu,
    cov=cov
)

filter_dynamics = linear_model(F, np.zeros(like=F))
true_dynamics = linear_model(F, cov)

# true initial state
x0 = np.array([[-1], [-1], [0.1], [0.1]])

# Initial prior is uniform distribution over map
n_pixels = len(x) * len(y)
prior = np.zeros_like(z) + 1 / n_pixels

steps = 100
for i in np.arange(steps):

    # predict
    prior * transition_distribution(F)

    # simulate measurement
    # 1. get true position
    
        # measure depth
        # add error
    
    # update
    prior * likelihood(measurement)
    

TypeError: eye() missing 1 required positional argument: 'N'

In [26]:
mesh

(array([[-1.        , -0.98994975, -0.9798995 , ...,  0.9798995 ,
          0.98994975,  1.        ],
        [-1.        , -0.98994975, -0.9798995 , ...,  0.9798995 ,
          0.98994975,  1.        ],
        [-1.        , -0.98994975, -0.9798995 , ...,  0.9798995 ,
          0.98994975,  1.        ],
        ...,
        [-1.        , -0.98994975, -0.9798995 , ...,  0.9798995 ,
          0.98994975,  1.        ],
        [-1.        , -0.98994975, -0.9798995 , ...,  0.9798995 ,
          0.98994975,  1.        ],
        [-1.        , -0.98994975, -0.9798995 , ...,  0.9798995 ,
          0.98994975,  1.        ]], shape=(200, 200)),
 array([[-1.        , -1.        , -1.        , ..., -1.        ,
         -1.        , -1.        ],
        [-0.98994975, -0.98994975, -0.98994975, ..., -0.98994975,
         -0.98994975, -0.98994975],
        [-0.9798995 , -0.9798995 , -0.9798995 , ..., -0.9798995 ,
         -0.9798995 , -0.9798995 ],
        ...,
        [ 0.9798995 ,  0.9798995 ,  

In [30]:
x = np.linspace(-1, 1, 3)

mesh = np.meshgrid(x, x)
mesh

(array([[-1.,  0.,  1.],
        [-1.,  0.,  1.],
        [-1.,  0.,  1.]]),
 array([[-1., -1., -1.],
        [ 0.,  0.,  0.],
        [ 1.,  1.,  1.]]))